In [1]:
from pyspark.sql import SparkSession
import psycopg2

In [2]:
spark=SparkSession.builder.appName('provider').getOrCreate()

25/04/30 10:23:37 WARN Utils: Your hostname, aayushgyawali resolves to a loopback address: 127.0.1.1; using 10.10.42.111 instead (on interface enp2s0)
25/04/30 10:23:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 10:23:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet("provider.parquet")
df1=spark.read.parquet("innetwork.parquet")

In [4]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="admin",
    host="localhost",
    port=5432
)

jdbc_url = "jdbc:postgresql://localhost:5432/postgres"
connection_properties = {
    "user": "postgres",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

In [5]:
df.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: short (nullable = true)
 |-- tin: string (nullable = true)



In [6]:
cur = conn.cursor()

# create_table_query = """
# DROP TABLE IF EXISTS provider_data;
# CREATE TABLE IF NOT EXISTS provider_data (
#     provider_group_id BIGINT,
#     npi BIGINT,
#     tin_type SMALLINT,
#     tin TEXT
# );
# """
# cur.execute(create_table_query)
# df.write.jdbc(url=jdbc_url,table="provider_data",mode="append", properties=connection_properties)

# conn.commit()

In [7]:
cur = conn.cursor()

create_table_query = """
DROP TABLE IF EXISTS innetwork_data;
CREATE TABLE IF NOT EXISTS innetwork_data (
    billing_code TEXT,
    billing_code_type TEXT,
    negotiation_arrangement TEXT,
    provider_group_id BIGINT,
    billing_class TEXT,
    billing_code_modifier TEXT[],
    negotiated_rate DOUBLE PRECISION,
    negotiated_type TEXT,
    service_code INTEGER[]
);
"""
cur.execute(create_table_query)
conn.commit()

df1.write.jdbc(url=jdbc_url,table="innetwork_data",mode="append", properties=connection_properties)

25/04/30 10:23:50 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [8]:
conn.commit()
cur.close()
conn.close()